In [2]:
import sys
sys.path.append('..')
from src.bq_client import run_query
import pandas as pd

- row counts for each table

In [3]:
df_counts = run_query("""
    SELECT 'users' AS table_name, COUNT(*) AS row_count 
    FROM `bigquery-public-data.thelook_ecommerce.users`
    UNION ALL
    SELECT 'orders', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.orders`
    UNION ALL
    SELECT 'order_items', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.order_items`
    UNION ALL
    SELECT 'products', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.products`
    UNION ALL
    SELECT 'events', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.events`
    UNION ALL
    SELECT 'inventory_items', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.inventory_items`
    UNION ALL
    SELECT 'distribution_centers', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.distribution_centers`
""")
df_counts

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,row_count
0,orders,124690
1,products,29120
2,inventory_items,487585
3,users,100000
4,order_items,180520
5,events,2416574
6,distribution_centers,10


- understanding events table

In [4]:
df = run_query("""
    SELECT *
    FROM `bigquery-public-data.thelook_ecommerce.events`
    LIMIT 5
""")
print(df.to_string(index=False))
print("\nCOLUMNS:", list(df.columns))

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


     id  user_id  sequence_number                           session_id                created_at      ip_address      city     state postal_code browser traffic_source     uri event_type
2394075     <NA>                3 6322d00d-98a7-411f-b0bc-61d2e5aba804 2020-11-15 06:14:00+00:00    188.90.5.218 São Paulo São Paulo   02220-000  Chrome        Adwords /cancel     cancel
1808851     <NA>                3 e11e4a16-8391-4da3-9b36-9f241a3c5a36 2023-03-17 01:15:00+00:00    73.79.126.98 São Paulo São Paulo   02675-031  Chrome          Email /cancel     cancel
2184038     <NA>                3 56863b99-abfe-4b94-ac7f-ad48f5dc21f7 2026-03-26 02:37:00+00:00  204.240.98.145 São Paulo São Paulo   02675-031  Chrome          Email /cancel     cancel
1625596     <NA>                3 1739888d-f30d-4e5c-a9c6-75a1d2338bcb 2023-12-01 11:54:00+00:00 147.156.226.136 São Paulo São Paulo   02675-031  Chrome        Adwords /cancel     cancel
1930285     <NA>                3 63b521c9-cad7-4698-a7a7-25f95a7

- event types and count

In [6]:
df = run_query("""
    SELECT event_type, COUNT(*) AS count
    FROM `bigquery-public-data.thelook_ecommerce.events`
    GROUP BY event_type
    ORDER BY count DESC
""")
print(df.to_string(index=False))

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


event_type  count
   product 841328
department 591416
      cart 591124
  purchase 180520
    cancel 124866
      home  87320


- null rates in event table

In [7]:
df = run_query("""
    SELECT
        ROUND(COUNTIF(user_id IS NULL) / COUNT(*) * 100, 2) AS user_id_null_pct,
        ROUND(COUNTIF(session_id IS NULL) / COUNT(*) * 100, 2) AS session_id_null_pct,
        ROUND(COUNTIF(created_at IS NULL) / COUNT(*) * 100, 2) AS created_at_null_pct,
        ROUND(COUNTIF(event_type IS NULL) / COUNT(*) * 100, 2) AS event_type_null_pct,
        ROUND(COUNTIF(ip_address IS NULL) / COUNT(*) * 100, 2) AS ip_address_null_pct
    FROM `bigquery-public-data.thelook_ecommerce.events`
""")
print(df.to_string(index=False))

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


 user_id_null_pct  session_id_null_pct  created_at_null_pct  event_type_null_pct  ip_address_null_pct
            46.54                  0.0                  0.0                  0.0                  0.0


- understanding orders table

In [8]:
df = run_query("""
    SELECT status, COUNT(*) AS count
    FROM `bigquery-public-data.thelook_ecommerce.orders`
    GROUP BY status
    ORDER BY count DESC
""")
print(df.to_string(index=False))

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    status  count
   Shipped  37342
  Complete  31176
Processing  24836
 Cancelled  18745
  Returned  12591


In [9]:
df = run_query("""
    SELECT
        MIN(created_at) AS earliest_order,
        MAX(created_at) AS latest_order
    FROM `bigquery-public-data.thelook_ecommerce.orders`
""")
print(df.to_string(index=False))

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


           earliest_order                     latest_order
2019-01-05 13:53:54+00:00 2026-06-24 00:48:06.823550+00:00


In [10]:
df = run_query("""
    SELECT 
        event_type,
        COUNT(*) AS total_events,
        COUNT(DISTINCT session_id) AS sessions_with_this_event,
        COUNT(DISTINCT user_id) AS unique_users
    FROM `bigquery-public-data.thelook_ecommerce.events`
    GROUP BY event_type
    ORDER BY total_events DESC
""")
df

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,event_type,total_events,sessions_with_this_event,unique_users
0,product,841328,680520,79987
1,department,591416,430608,79987
2,cart,591124,430316,79987
3,purchase,180520,180520,79987
4,cancel,124866,124866,0
5,home,87320,87320,62991


In [11]:
df = run_query("""
    SELECT
        CASE WHEN user_id IS NULL THEN 'anonymous' ELSE 'authenticated' END AS user_type,
        COUNT(*) AS event_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
    FROM `bigquery-public-data.thelook_ecommerce.events`
    GROUP BY user_type
""")
df

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,user_type,event_count,pct_of_total
0,authenticated,1291824,53.46
1,anonymous,1124750,46.54


In [15]:
# Find a real user with lots of events to use for manual verification tomorrow
df = run_query("""
    SELECT 
        user_id,
        COUNT(*) AS event_count,
        MIN(created_at) AS first_event,
        MAX(created_at) AS last_event
    FROM `bigquery-public-data.thelook_ecommerce.events`
    WHERE user_id IS NOT NULL
    GROUP BY user_id
    ORDER BY event_count DESC
    LIMIT 10
""")
df

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,user_id,event_count,first_event,last_event
0,83404,170,2026-02-23 16:00:48+00:00,2026-06-08 02:14:21+00:00
1,13915,170,2024-06-13 01:22:42+00:00,2025-11-12 16:06:32+00:00
2,87498,161,2021-05-28 16:18:54+00:00,2025-03-28 04:19:04+00:00
3,2764,156,2022-03-17 14:13:53+00:00,2026-01-12 04:28:51+00:00
4,43726,148,2023-07-13 01:45:54+00:00,2026-01-28 19:03:59+00:00
5,59030,139,2025-04-22 04:32:44+00:00,2026-05-07 03:13:03+00:00
6,53355,139,2023-12-12 09:50:04+00:00,2025-01-07 10:38:10+00:00
7,97861,139,2021-08-10 15:36:31+00:00,2025-10-12 22:09:38+00:00
8,70543,139,2024-12-08 05:03:37+00:00,2025-12-30 11:01:33+00:00
9,83682,134,2026-04-25 08:46:06+00:00,2026-06-23 22:15:34+00:00
